# Clase 3: Redes Neuronales Convolucionales (CNN) y Vision Artificial

## Caso de Estudio: Clasificacion de Imagenes - Perros vs Gatos

**Modulo:** Cientifico de Datos e Inteligencia Artificial Aplicada
**Unidad:** 1
**Tecnologias:** Python, TensorFlow, Keras, NumPy, Matplotlib, Pillow, Scikit-Learn
**Dataset:** [Dogs vs Cats (Kaggle)](https://www.kaggle.com/datasets/salader/dogs-vs-cats)

> **Version para clase en vivo:** las celdas de codigo estan vacias (solo con una guia en
> comentario). El desarrollo se hace junto con los estudiantes durante la sesion. La explicacion
> completa de cada paso esta en las celdas de markdown.


## Objetivo de esta practica

En la Clase 1 construimos una ANN totalmente conectada (Dense) y en la Clase 2 aprendimos a
entrenarla bien (optimizadores, regularizacion, callbacks). Esas redes recibian **datos tabulares**:
filas y columnas, sin ninguna relacion espacial entre features.

Hoy cambiamos de tipo de dato: vamos a trabajar con **imagenes**. Una imagen no es una lista plana
de numeros, es una **matriz con estructura espacial** (un pixel se parece a sus vecinos, un borde es
un patron local, una oreja de gato aparece siempre con la misma forma sin importar en que parte de
la foto este). Una red Dense ignora por completo esa estructura. Una **Red Neuronal Convolucional
(CNN)** esta disenada especificamente para aprovecharla.

El objetivo de este notebook no es solo entrenar un clasificador que funcione, sino **ver con los
propios ojos como se transforma una imagen** en cada etapa del proceso:

1. Como cambia al redimensionarla y normalizarla.
2. Como cambia al aplicarle *data augmentation* (rotaciones, flips, zoom).
3. Como cambia, capa por capa, al pasar por los filtros convolucionales de la red (los llamados
   *mapas de activacion*).

**Flujo de la clase:**

`dataset de Kaggle -> exploracion visual -> pipeline de preprocesamiento (visualizado paso a paso)
-> arquitectura CNN (explicada capa por capa) -> entrenamiento -> diagnostico -> evaluacion
-> mapas de activacion antes/despues de entrenar`


## 0) El Dataset: Dogs vs Cats (Kaggle)

Para esta clase usamos **[Dogs vs Cats](https://www.kaggle.com/datasets/salader/dogs-vs-cats)**,
una version ya organizada en carpetas de la competencia clasica de Kaggle. Trae **25000 imagenes**
en total, ya separadas en train/test y en subcarpetas por clase (esto es clave: Keras puede leer
un dataset de imagenes directamente si esta organizado asi).

Como en la Clase 2, **el dataset no se versiona en el repositorio** (son cientos de MB de imagenes).
Cada estudiante lo descarga en su propio entorno.

### Opcion A: descarga manual

1. Crear cuenta en https://www.kaggle.com
2. Entrar a https://www.kaggle.com/datasets/salader/dogs-vs-cats y descargar el `.zip`.
3. Descomprimirlo dentro de la carpeta de esta clase, de forma que quede asi:

```
clase_03_cnn_vision_artificial/
|-- data/
|   |-- train/
|   |   |-- cats/   (10000 imagenes, cat.0.jpg ... cat.9999.jpg)
|   |   |-- dogs/   (10000 imagenes, dog.0.jpg ... dog.9999.jpg)
|   |-- test/
|       |-- cats/   (2500 imagenes)
|       |-- dogs/   (2500 imagenes)
|-- U1_C3_cnn_vision_artificial_v1.ipynb
```

### Opcion B: API oficial de Kaggle

```bash
pip install kaggle
# Token: https://www.kaggle.com/settings -> API -> Create New Token
# Windows: C:\Users\<usuario>\.kaggle\kaggle.json
# Linux/macOS: ~/.kaggle/kaggle.json   (chmod 600 ~/.kaggle/kaggle.json)
kaggle datasets download -d salader/dogs-vs-cats -p data --unzip
```

> **Nota:** esta carpeta de clase ya tiene el dataset descargado y organizado en `data/train/` y
> `data/test/`, exactamente con esa estructura. No hace falta volver a descargarlo.

> Recuerda anadir `data/` al `.gitignore` del repositorio para no subir las imagenes.


In [ ]:
%pip install tensorflow matplotlib pillow numpy scikit-learn scipy

In [ ]:
import os
import random

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices('GPU'))


## 1) Explorando el dataset

Antes de escribir una sola linea de modelo, siempre exploramos los datos. Con imagenes preguntamos
cosas distintas a las de un dataset tabular: ¿cuantas imagenes hay por clase?, ¿estan balanceadas?,
¿todas tienen el mismo tamano?, ¿que aspecto tienen?


In [ ]:
# EN CLASE: definir DATA_DIR, TRAIN_DIR, TEST_DIR y CLASS_NAMES = ["cats", "dogs"]
# y recorrer cada carpeta (train/cats, train/dogs, test/cats, test/dogs) contando
# cuantas imagenes hay en cada una con os.listdir().


Las imagenes vienen en tamanos distintos (una foto de camara, no un dataset sintetico). Eso es
justo uno de los primeros problemas que resuelve nuestro pipeline: **toda imagen debe entrar a la
red con el mismo tamano fijo**, porque las capas Dense finales necesitan un numero fijo de entradas.
Vamos a confirmarlo mirando el tamano real de unas cuantas imagenes.


In [ ]:
# EN CLASE: tomar 5 archivos de train/cats, abrirlos con PIL.Image.open() e
# imprimir su tamano (img.size) y modo de color (img.mode) para confirmar que varian.


In [ ]:
# EN CLASE: escribir una funcion mostrar_grid(carpeta, n, titulo) que tome n imagenes
# al azar de una carpeta (random.sample) y las muestre en una grilla de matplotlib con
# su tamano en el titulo de cada subplot. Usarla sobre train/cats y train/dogs.


## 2) Por que una CNN y no una red Dense

Si aplanaramos una imagen de 150x150 pixeles a color en un vector para una red Dense, tendriamos
`150 * 150 * 3 = 67500` entradas, y la primera capa Dense necesitaria un peso distinto para **cada
pixel en cada posicion**. Dos problemas graves:

- **Demasiados parametros**: el modelo se sobreajusta facilmente y es carisimo de entrenar.
- **Ninguna nocion de "vecindad"**: la red no sabe que un pixel esta al lado de otro. Si un gato se
  mueve 10 pixeles a la derecha en la foto, para una red Dense es una entrada completamente distinta.

Una **capa convolucional** resuelve esto de forma elegante: en vez de conectar cada pixel a cada
neurona, desliza un **filtro (kernel)** pequeno (por ejemplo 3x3) sobre toda la imagen, aplicando
la **misma operacion** en cada posicion. Esto le da a la CNN dos propiedades clave:

- **Menos parametros**: un filtro de 3x3x3 tiene solo 27 pesos, y se reutiliza en toda la imagen
  (*weight sharing*).
- **Invarianza a traslacion**: si el filtro aprende a detectar "orejas puntiagudas", las detecta
  sin importar en que parte de la foto aparezcan.

### Vocabulario que vamos a usar

- **Filtro / Kernel**: una matriz pequena de pesos (ej. 3x3) que se desliza sobre la imagen.
- **Mapa de activacion (feature map)**: el resultado de aplicar un filtro a toda la imagen.
- **Stride**: cuantos pixeles se mueve el filtro en cada paso.
- **Padding**: si se rellenan los bordes para que el mapa de salida mantenga el tamano de entrada.
- **Pooling**: una operacion que reduce el tamano espacial del mapa de activacion, quedandose con
  la informacion mas relevante (ej. `MaxPooling` se queda con el valor maximo de cada bloque).

La mejor forma de entender un filtro convolucional es **verlo actuar sobre una imagen real**.


### 2.1) Un filtro convolucional en accion (implementado a mano)

Antes de dejar que Keras haga esto automaticamente dentro del modelo, vamos a aplicar **un filtro
convolucional manualmente**, con NumPy puro, sobre una imagen real. Usamos un filtro clasico de
deteccion de bordes verticales para que el efecto sea muy visible.


In [ ]:
# EN CLASE: cargar una imagen de ejemplo (train/cats), convertirla a escala de grises
# y redimensionarla a 200x200. Definir el kernel de bordes verticales:
#   [[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]]
# y aplicarlo con scipy.signal.convolve2d(imagen, kernel, mode="valid").
# Mostrar la imagen original y el mapa de activacion resultante lado a lado.


Observa dos cosas en el resultado:

1. **El contenido cambio**: el filtro "resalta" los bordes verticales (donde la imagen pasa de
   claro a oscuro horizontalmente) y apaga las zonas planas. Esto es exactamente lo que hace cada
   filtro dentro de una capa `Conv2D`, solo que en una CNN los valores del filtro **no los elegimos
   nosotros**: la red los aprende durante el entrenamiento.
2. **El tamano cambio**: la imagen de entrada era 200x200 y el mapa de activacion salio en 198x198.
   Esto es el efecto del `padding="valid"` (sin relleno): un filtro de 3x3 "pierde" un pixel de
   borde por cada lado. En Keras podemos evitarlo con `padding="same"`.

### Pooling: reducir el tamano quedandonos con lo importante

Despues de una o varias convoluciones, casi siempre aplicamos **MaxPooling**: dividimos el mapa de
activacion en bloques (ej. 2x2) y nos quedamos con el valor maximo de cada bloque. Esto reduce el
tamano espacial a la mitad y hace que la red sea mas eficiente y algo mas robusta a pequenos
desplazamientos. Vamos a aplicarlo, tambien a mano, sobre el mapa de activacion que acabamos de
calcular.


In [ ]:
# EN CLASE: escribir max_pooling_manual(mapa, tamano_bloque=2) que divida el mapa
# de activacion en bloques de 2x2 y se quede con el maximo de cada bloque (doble for + .max()).
# Aplicarlo sobre el mapa de activacion de la celda anterior y graficar en una fila de 3:
# imagen original -> mapa tras convolucion -> mapa tras pooling, mostrando el shape de cada uno.


## 3) Pipeline de preprocesamiento: viendo la imagen cambiar paso a paso

Ya entendimos que hace un filtro convolucional. Ahora armemos el **pipeline completo** que le
prepara cada imagen a la red, siguiendo la misma imagen de ejemplo en cada etapa:

1. **Carga**: leer el archivo desde disco (tamano original, variable).
2. **Resize**: llevarla a un tamano fijo, el mismo para todas las imagenes del dataset.
3. **Conversion a array + normalizacion**: pasar de pixeles en el rango `[0, 255]` a valores en
   `[0, 1]` (redes neuronales entrenan mejor con entradas pequenas y centradas).
4. **Tensor final**: la forma exacta que va a recibir la primera capa de la CNN.


In [ ]:
# EN CLASE: definir IMG_SIZE = (150, 150) y, sobre la imagen de ejemplo:
#   1) abrirla con PIL (tamano original)
#   2) .convert("RGB").resize(IMG_SIZE)
#   3) convertirla a np.array(dtype=np.float32) y normalizar dividiendo entre 255.0
#   4) agregar la dimension de batch con np.expand_dims(..., axis=0)
# Imprimir shape y rango de valores en cada paso, y graficar las 3 primeras etapas lado a lado.


> **Por que normalizar:** los pixeles en `[0, 255]` producen activaciones muy grandes en la
> primera capa, lo que hace el entrenamiento inestable (parecido al problema de escalas distintas
> que vimos con `StandardScaler` en el Titanic, en la Clase 2). Al llevar los valores a `[0, 1]`
> el gradiente fluye mejor y el entrenamiento converge mas rapido y de forma mas estable.

## 4) Data Augmentation: multiplicando la variedad de los datos

Con "solo" 20000 imagenes de entrenamiento, una CNN puede memorizar detalles especificos de las
fotos (overfitting), tal como vimos con la ANN del Titanic en la Clase 2. Una tecnica muy efectiva
en vision artificial es el **data augmentation**: generar versiones ligeramente modificadas de cada
imagen (rotada, volteada, con zoom) en cada epoca de entrenamiento. La red nunca ve exactamente la
misma imagen dos veces, lo que la obliga a aprender **patrones generales** ("forma de oreja de
gato") en vez de memorizar pixeles especificos.

Vamos a aplicar cada transformacion por separado sobre la misma imagen para ver claramente su
efecto, y luego todas juntas y aplicadas varias veces para ver la variabilidad que introducen.


In [ ]:
# EN CLASE: crear una capa por transformacion:
#   layers.RandomFlip("horizontal"), layers.RandomRotation(0.2),
#   layers.RandomZoom(0.2), layers.RandomContrast(0.3)
# y graficar, en una fila, el resultado de aplicar cada una (por separado) sobre la
# misma imagen de ejemplo (el tensor con batch dimension de la celda anterior).


In [ ]:
# EN CLASE: combinar las 4 transformaciones en un keras.Sequential llamado
# data_augmentation, y aplicarlo 8 veces (training=True) sobre la MISMA imagen para
# graficar una grilla de 8 variantes distintas generadas a partir de una sola foto.


> **Importante:** el data augmentation **solo se aplica al set de entrenamiento**. Nunca se
> aumenta el set de validacion ni el de test: necesitamos evaluar el modelo sobre datos reales,
> no sobre variaciones artificiales.

## 5) Construyendo los datasets con `tf.data`

Con el pipeline ya entendido paso a paso, ahora dejamos que Keras lo aplique automaticamente a las
25000 imagenes usando `image_dataset_from_directory`, que:

- Lee las imagenes directamente desde las carpetas `cats/` y `dogs/` (la subcarpeta define la
  etiqueta automaticamente).
- Las redimensiona todas a `IMG_SIZE`.
- Las agrupa en batches.
- Separa una porcion del set de entrenamiento como set de **validacion** (para diagnosticar
  overfitting durante el entrenamiento, igual que hicimos en la Clase 2).


In [ ]:
# EN CLASE: BATCH_SIZE = 32. Usar keras.utils.image_dataset_from_directory() tres veces:
#   - train_ds: sobre TRAIN_DIR, validation_split=0.2, subset="training", seed=SEED
#   - val_ds:   sobre TRAIN_DIR, validation_split=0.2, subset="validation", seed=SEED
#   - test_ds:  sobre TEST_DIR, shuffle=False
# label_mode="binary", image_size=IMG_SIZE, batch_size=BATCH_SIZE en los tres.
# Guardar class_names = train_ds.class_names e imprimirlo.
# Aplicar .prefetch(tf.data.AUTOTUNE) a los tres datasets.


## 6) Arquitectura de la CNN

Vamos a construir la red como una secuencia de **bloques convolucionales**: cada bloque es
`Conv2D -> ReLU -> MaxPooling`. Cada bloque reduce el tamano espacial de la imagen (gracias al
pooling) y aumenta el numero de filtros (de 32 a 128), siguiendo un patron muy comun en vision
artificial: **al avanzar en profundidad, la red ve "menos espacio pero mas conceptos"**.

- **Capas `Rescaling` y `data_augmentation`** al inicio: el preprocesamiento que hicimos a mano
  ahora queda *dentro* del modelo, para que se aplique automaticamente a cualquier imagen nueva.
- **Bloques `Conv2D + MaxPooling2D`**: extraen features cada vez mas abstractas (bordes -> texturas
  -> formas -> partes del animal).
- **`Flatten` / `GlobalAveragePooling2D`**: convierte el mapa de activacion 3D final en un vector 1D
  para poder conectarlo a capas Dense.
- **`Dropout`**: regularizacion, igual que en la Clase 2, para reducir overfitting.
- **Capa de salida `Dense(1, activation="sigmoid")`**: como es clasificacion **binaria**
  (cat vs dog), una sola neurona con sigmoide nos da una probabilidad entre 0 y 1.


In [ ]:
# EN CLASE: construir_modelo() usando la API funcional de Keras:
#   Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
#   -> data_augmentation -> Rescaling(1/255)
#   -> [Conv2D(32) + MaxPooling2D] -> [Conv2D(64) + MaxPooling2D]
#   -> [Conv2D(128) + MaxPooling2D] -> [Conv2D(128) + MaxPooling2D]
#   -> Flatten -> Dropout(0.5) -> Dense(256, relu) -> Dense(1, sigmoid)
# Crear el modelo y llamar model.summary().


### Como leer el `model.summary()`

- La columna `Output Shape` muestra como se **reduce el alto y el ancho** en cada `MaxPooling2D`
  (150 -> 75 -> 37 -> 18 -> 9) mientras el numero de **canales (filtros) crece** (32 -> 64 -> 128
  -> 128). Es literalmente la imagen "encogiendose" mientras la red extrae mas conceptos de ella.
- `Param #` en las capas `Conv2D` es pequeno comparado con las capas `Dense`: la mayoria de los
  parametros del modelo estan concentrados en la capa `Dense(256)`, justo despues de aplanar. Esto
  confirma por que aplanar una imagen entera para una red 100% Dense (como discutimos en la
  seccion 2) seria muchisimo mas costoso.


In [ ]:
# EN CLASE: model.compile() con optimizer=keras.optimizers.Adam(1e-3),
# loss="binary_crossentropy", metrics=["accuracy"].
# Definir callbacks_list con EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
# ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6) y
# ModelCheckpoint("mejor_modelo_dogs_vs_cats.keras", monitor="val_loss", save_best_only=True).


## 7) Mapas de activacion ANTES de entrenar

Antes de entrenar, la red tiene pesos **aleatorios**. Aun asi podemos "abrirla" y ver que produce
cada capa convolucional al pasarle nuestra imagen de ejemplo. Vamos a construir un modelo auxiliar
que expone las salidas intermedias de cada bloque convolucional, para visualizar como la imagen se
va transformando capa por capa dentro de la red.


In [ ]:
# EN CLASE: escribir visualizar_mapas_activacion(modelo, imagen_tensor, titulo_extra="").
# Idea: tomar las salidas de todas las capas Conv2D del modelo (keras.Model(inputs=modelo.input,
# outputs=[esas salidas])), predecir sobre la imagen de ejemplo, y graficar los primeros 6 filtros
# de cada capa en una grilla (filas = capas, columnas = filtros).
# Llamarla sobre el modelo recien creado (sin entrenar) con la imagen de ejemplo.


Con pesos aleatorios, los mapas de activacion no muestran ningun patron reconocible: son
practicamente ruido con formas vagas. Esto tiene sentido, los filtros todavia no han aprendido
nada. Vamos a entrenar la red y, al final del notebook, **repetir exactamente esta misma
visualizacion** para comparar. La diferencia va a ser muy clara.

## 8) Entrenamiento

Entrenamos el modelo sobre `train_ds`, monitoreando `val_ds` en cada epoca. Los callbacks que
definimos (`EarlyStopping`, `ReduceLROnPlateau`, `ModelCheckpoint`) son los mismos conceptos que
vimos en la Clase 2: dejar que el propio entrenamiento decida cuando detenerse y cuando bajar el
learning rate, en vez de elegir el numero de epocas a mano.

> Ajusta `EPOCHS` segun el tiempo disponible en clase. Con `EarlyStopping` activo, el entrenamiento
> se detiene solo si deja de mejorar, asi que un numero alto es seguro (no se van a completar todas
> las epocas si el modelo ya convergio).


In [ ]:
# EN CLASE: EPOCHS = 20 y model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
# callbacks=callbacks_list). Guardar el resultado en la variable history.


## 9) Curvas de entrenamiento: diagnostico

Igual que en la Clase 2, la mejor forma de saber si el modelo esta aprendiendo bien es graficar
`accuracy` y `loss` de entrenamiento contra validacion.

### Como leer estas graficas

- Si **train** mejora y **val** se estanca o empeora: overfitting. La red esta memorizando el set
  de entrenamiento. El data augmentation y el dropout que agregamos ayudan a mitigar esto.
- Si **ambas curvas** son malas y no mejoran: underfitting. El modelo es muy simple, hay que
  entrenar mas epocas, aumentar filtros o bajar la regularizacion.
- Si ambas mejoran juntas y convergen: la red esta generalizando bien.


In [ ]:
# EN CLASE: tomar history.history (diccionario con "loss", "val_loss", "accuracy",
# "val_accuracy") y graficar dos subplots: accuracy (train vs val) y loss (train vs val),
# ambos contra el numero de epoca.


## 10) Evaluacion en el conjunto de test

Las 5000 imagenes de `data/test/` nunca fueron usadas ni para entrenar ni para validar durante el
entrenamiento. Es la medida mas honesta de que tan bien generaliza el modelo.


In [ ]:
# EN CLASE: model.evaluate(test_ds) para test_loss y test_accuracy.
# Construir y_true concatenando las etiquetas de test_ds, y y_pred_prob con model.predict(test_ds).
# Umbralizar y_pred_prob >= 0.5 para obtener y_pred.
# Imprimir classification_report(y_true, y_pred, target_names=class_names) y graficar la
# matriz de confusion con ConfusionMatrixDisplay.


### Prediciones visuales

Nada explica mejor los aciertos y errores del modelo que verlos sobre imagenes reales. Tomamos un
batch de test y mostramos la prediccion junto a cada imagen: en verde si acerto, en rojo si fallo.


In [ ]:
# EN CLASE: tomar un batch de test_ds (test_ds.take(1)), predecir con el modelo,
# y graficar 8 imagenes con su etiqueta real, la prediccion y la probabilidad, coloreando
# el titulo en verde si acerto y en rojo si fallo.


## 11) Mapas de activacion DESPUES de entrenar: cerrando el circulo

Repetimos exactamente la misma visualizacion de la seccion 7, sobre la misma imagen de ejemplo,
pero ahora con los pesos **ya entrenados**. Compara este resultado con el de antes: ahora los
filtros de las primeras capas suelen resaltar bordes y texturas de forma mucho mas nitida, y los
de las capas mas profundas activan regiones que corresponden a partes reconocibles del animal
(orejas, ojos, hocico) en vez de ruido aleatorio.


In [ ]:
# EN CLASE: volver a llamar visualizar_mapas_activacion(model, ejemplo_tensor,
# titulo_extra="(pesos YA entrenados)") y comparar visualmente contra la version de la seccion 7.


## Resumen: el recorrido de la imagen

En este notebook seguimos **una sola imagen de ejemplo** a traves de todo el pipeline, para que el
proceso completo quedara visible en cada paso:

1. Imagen original (tamano variable) -> redimensionada a `150x150` -> normalizada a `[0, 1]`.
2. Data augmentation: la misma imagen generando variantes distintas en cada epoca (flip, rotacion,
   zoom, contraste).
3. Filtros convolucionales aplicados a mano (deteccion de bordes) y luego dentro del modelo real.
4. Mapas de activacion capa por capa, antes de entrenar (ruido) y despues de entrenar (patrones
   reconocibles).

Esa es, en esencia, la idea completa de una CNN: **una cadena de transformaciones aprendidas** que
convierte pixeles crudos en una prediccion.

## Actividad rapida

1. Cambia `IMG_SIZE` a `(100, 100)` y vuelve a correr el notebook. ¿Como cambia el tiempo de
   entrenamiento? ¿Como cambia el accuracy en test?
2. Agrega o quita un bloque `Conv2D + MaxPooling2D` en `construir_modelo()`. ¿El modelo mejora o
   empeora? ¿Por que?
3. Modifica los parametros de `data_augmentation` (por ejemplo, sube `RandomRotation` a `0.4`).
   Vuelve a correr la celda de la seccion 4 que muestra las 8 variantes. ¿En que punto la imagen
   se distorsiona tanto que ya no ayuda a entrenar?
4. **Bonus (transfer learning):** reemplaza los bloques convolucionales por un modelo preentrenado
   como `keras.applications.MobileNetV2(weights="imagenet", include_top=False)` congelado, y
   compara el accuracy y el tiempo de entrenamiento contra la CNN entrenada desde cero.
